# MobileNetV2 --- SNIP-it and WANDA at n=3

Completes MobileNetV2's criterion coverage. Magnitude is already done at n=3
on a separate box; this notebook adds the other two criteria under the same
finalised protocol: **both arms at SGD 0.1**.

## Why both arms at 0.1

MobileNetV2 inherited the ResNet recipe (I.P. 0.01 / BaCP 0.1) untuned. A
learning-rate probe showed 0.01 is wrong for this model's I.P. arm above 0.5
sparsity --- at 0.99 it collapses to chance (10.02) while 0.1 reaches 78.68.
Both arms therefore run at 0.1 here, which is each arm's measured optimum at
the sparsities that matter (BaCP at 0.5 was also tried and is 7.2 points
worse than 0.1).

## WANDA caveat --- read before reporting the WANDA row

`pruning_factory`'s WANDA path falls back to plain magnitude ranking for any
layer whose activation statistic width does not match the flattened weight,
and its own comment notes grouped and depthwise Conv2d hit this
*deterministically* (`F.unfold` is group-unaware). MobileNetV2 has 17
depthwise layers of 53. Those layers are small --- roughly 64k of 2.2M
prunable weights --- so most of the pool still receives a genuine WANDA
score, but the row is not purely WANDA. The health cell at the bottom prints
the fallback fraction so this can be stated rather than glossed.

## Ordering

Criterion-major, not seed-major: an interrupted run should leave SNIP
complete at n=3 rather than both criteria stranded at n=1, since a delta
without seeds is not a usable number. Sparsities are the paper's grid exactly (0.95/0.97/0.99/0.999), so
MobileNetV2's rows line up with the ResNet/VGG table.

Safe to interrupt and re-run --- a recorded cell is skipped.


In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Weights + preflight

Halts before any GPU time if the model is not actually pretrained.

In [ ]:
nb.fetch_imagenet_weights('mobilenet_v2')
nb.preflight('mobilenet_v2', num_classes=nb.FAMILIES['mobilenet_v2']['base']['num_classes'])

## Dense baselines

Every sparse cell resolves its checkpoint from the dense run of the SAME seed.

In [ ]:
MODEL, GPU, SEEDS = 'mobilenet_v2', 0, (1, 2, 3)
dense = [nb.make_cell(MODEL, 'dense', seed=s) for s in SEEDS]
nb.run_group(dense, gpu=GPU)

## Plan

Built and sanity-checked before any training.

In [ ]:
SPARSITIES = (0.95, 0.97, 0.99, 0.999)   # the paper's grid

plan = []
for pruner in ('snip', 'wanda'):
    for sp in SPARSITIES:
        for seed in SEEDS:
            plan.append(nb.make_cell(MODEL, 'prune', seed=seed, pruner=pruner,
                                     sparsity=sp, learning_rate=0.1))
            plan.append(nb.make_cell(MODEL, 'bacp', seed=seed, pruner=pruner,
                                     sparsity=sp))

for c in plan:
    assert c['config']['learning_rate'] == 0.1, c['config']['learning_rate']

n_ip = sum(1 for c in plan if c['rung'] == 'static-prune')
print(f'{len(plan)} cells = {n_ip} I.P. + {len(plan) - n_ip} BaCP, both arms lr=0.1')
print(f'est ~{(n_ip * 4.6 + (len(plan) - n_ip) * 16.5) / 60:.1f} h')
assert nb.sanity_check(plan), 'sanity check failed'

## Run

One row per epoch. `results.csv` is rewritten after every cell.

In [ ]:
nb.run_group(plan, gpu=GPU)

## Results

In [ ]:
import json, glob, os, statistics as st
root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k and 'mobilenet' in k:
        acc.setdefault(k, r.get('test_acc_exact_pct') or r.get('test_acc_pct'))

def cell(arm, pruner, sp):
    xs = [acc.get(f'static.{arm}.{MODEL}.cifar10.s{sp}.{pruner}.seed{n}') for n in SEEDS]
    xs = [x for x in xs if x is not None]
    if not xs:
        return None
    return st.mean(xs), (st.stdev(xs) if len(xs) > 1 else 0.0), len(xs)

for pruner in ('magnitude', 'snip', 'wanda'):
    print(f'
--- {pruner} ---')
    print(f'{"sp":>7} | {"I.P.":>16} | {"BaCP":>16} | {"delta":>7}')
    for sp in SPARSITIES:
        a, b = cell('prune', pruner, sp), cell('bacp', pruner, sp)
        fa = f'{a[0]:.2f}+-{a[1]:.2f}(n={a[2]})' if a else '     --     '
        fb = f'{b[0]:.2f}+-{b[1]:.2f}(n={b[2]})' if b else '     --     '
        d = f'{b[0]-a[0]:+.2f}' if (a and b) else '   --'
        print(f'{sp:>7} | {fa:>16} | {fb:>16} | {d:>7}')

## Health --- WANDA depthwise fallback

How much of the WANDA row is actually magnitude ranking.

In [ ]:
import torch, torch.nn as nn
from model_factory import initialize_model_components
from pruning_factory import layer_check, set_prunable_scope

set_prunable_scope(prune_task_head=True, prune_embeddings=False)
m = initialize_model_components(MODEL, pretrained=False, dyrelu_en=False,
                                dyrelu_phasing_en=False, num_classes=10)['model']
prunable = {n: p for n, p in m.named_parameters() if layer_check(n, p)}
dw = {n for n, mod in m.named_modules() if isinstance(mod, nn.Conv2d) and mod.groups > 1}
dw_w = sum(p.numel() for n, p in prunable.items()
           if any(n.startswith(d + '.') for d in dw))
tot = sum(p.numel() for p in prunable.values())
print(f'depthwise/grouped conv layers : {len(dw)}')
print(f'weights in those layers       : {dw_w:,} of {tot:,} prunable  ({100*dw_w/tot:.2f}%)')
print()
print('Those layers fall back to magnitude ranking under WANDA')
print('(F.unfold is group-unaware). The remaining '
      f'{100*(1-dw_w/tot):.2f}% receives a genuine WANDA score.')